# Investment sandbox

A small agent-based model of an electricity market, in which firms decide what to build. This notebook runs it three times on the same demand and weather: with firms that ignore risk, with cautious firms, and with cautious firms and a government tender. The repository's README explains how the model works.

The model is illustrative. It follows one region with a stylised set of power stations, and nothing it produces is a forecast.

In [ ]:
import importlib.util, subprocess, sys

# Installs the package when the notebook runs somewhere it is not already present, such as Colab
if importlib.util.find_spec("investment_sandbox") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/MarkusMannheim/investment_sandbox.git"])

import pandas as pd
from investment_sandbox.config import load_settings
from investment_sandbox.simulate import compare
from investment_sandbox import explore, plots

pd.options.display.float_format = "{:,.1f}".format

## Settings

Change a setting, then run this cell and the cells below it again.

In [ ]:
CAUTION = 1.0                  # how strongly the firms dislike risk; at 0 they build whatever breaks even
DEMAND_PATH = "central"        # the demand growth that arrives: "low", "central" or "high"
SEED = 1                       # draws the weather year for each year of the run
BIDDERS = ["ocgt", "ccgt", "battery_4h", "battery_8h"]    # the technologies that can bid in the tender
CONTRACT_YEARS = 12            # how long a contract fixes a plant's income
RELIABILITY_STANDARD = 0.002   # the most unmet demand the government accepts in the forecast, % of demand

settings = load_settings({
    "investment": {"caution": CAUTION},
    "tender": {"technologies": BIDDERS, "contract_years": CONTRACT_YEARS,
               "reliability_standard": RELIABILITY_STANDARD / 100},
})
runs = compare(settings, seed=SEED, path=DEMAND_PATH)

## Results

The table sums up each run over its 20 years. The bill is what consumers pay for energy: supply at the spot price and unmet demand at the price cap. Underwriting is what the tender's contracts cost whoever stands behind them, which is the fixed sums paid to the plants less what those plants earned in the market. The resource cost is what supplying the electricity uses up: fuel, upkeep, capital and unmet demand at the price cap.

In [ ]:
pd.DataFrame(explore.headline(runs))

## Charts

The first chart shows the megawatts committed in each year, by technology. Hatched bars are plant bought by the tender.

In [ ]:
plots.committed(runs)

The next chart shows the plant in service in each year.

In [ ]:
plots.capacity(runs)

This chart shows the energy each technology supplied in each year.

In [ ]:
plots.generation(runs)

This chart shows the average price paid for a megawatt hour in each year.

In [ ]:
plots.prices(runs)

This chart shows unmet demand in each year as a share of demand. The dotted line is the reliability standard.

In [ ]:
plots.reliability(runs)

This chart shows the three sums of money for each run, added up over the 20 years.

In [ ]:
plots.costs(runs)

The last chart shows the hurdle each technology faced in one year's forecast, in the run with cautious firms and no tender. Each bar is the cost of owning the plant with the firm's caution added, and the upright mark is the income the plant expects. A firm builds when the mark sits past the end of the bar. The second bar of each pair is the hurdle under a contract. Change `YEAR` and run the cell again.

In [ ]:
YEAR = 2040
plots.hurdle(runs["cautious"], YEAR, CONTRACT_YEARS)